In [1]:
import os
import torch

#Detectamos la versión nativa de PyTorch y CUDA en tu Colab actual
version_torch = torch.__version__.split('+')[0]
version_cuda = "cu121" # Versión actual de CUDA en Colab
os.environ['TORCH'] = version_torch

print(f"Instalando aceleradores para PyTorch {version_torch} y CUDA {version_cuda}...")

# Instalación condicional de PyG y los motores C++ para GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Uninstall ALL potentially problematic libraries first to ensure a clean install
# This includes torch-geometric itself.
!pip uninstall -y torch-geometric pyg_lib torch_sparse torch_scatter

if device.type == 'cuda':
    os.environ['CUDA'] = version_cuda
    print("Instalando torch-geometric, pyg-lib, torch-sparse, torch-scatter para GPU...")
    !pip install torch-geometric
    !pip install pyg_lib torch_sparse torch_scatter -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html
else:
    os.environ['CUDA'] = 'cpu'
    print("Instalando torch-geometric, pyg-lib, torch-sparse, torch-scatter para CPU...")
    !pip install torch-geometric
    !pip install pyg_lib torch_sparse torch_scatter -f https://data.pyg.org/whl/torch-${TORCH}+cpu.html --quiet

Instalando aceleradores para PyTorch 2.3.0 y CUDA cu121...
Found existing installation: torch-geometric 2.7.0
Uninstalling torch-geometric-2.7.0:
  Successfully uninstalled torch-geometric-2.7.0
Found existing installation: pyg_lib 0.4.0+pt23cpu
Uninstalling pyg_lib-0.4.0+pt23cpu:
  Successfully uninstalled pyg_lib-0.4.0+pt23cpu
Found existing installation: torch_sparse 0.6.18+pt23cpu
Uninstalling torch_sparse-0.6.18+pt23cpu:
  Successfully uninstalled torch_sparse-0.6.18+pt23cpu
Found existing installation: torch_scatter 2.1.2+pt23cpu
Uninstalling torch_scatter-2.1.2+pt23cpu:
  Successfully uninstalled torch_scatter-2.1.2+pt23cpu
Instalando torch-geometric, pyg-lib, torch-sparse, torch-scatter para CPU...
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)


In [2]:
import pandas as pd
from torch_geometric.data import download_url, extract_zip
import torch

# Link Prediction

* Forma heurística: Analiza qué nodos tienen muchos vecinos en común y propone enlazarlos.
* Machine Learning: Dada una red G=(V,E), definimos enlaces positivos $E_p \subseteq E$ y enlaces negativos de entrenamiento $E_n | E_n \cap E = \emptyset$ para generar nuevos *node embeddings* para $G'=(V,E\cup E_n)$

¿Por qué se hace ese enfoque en ML? Así como vimos en Infomap, Louvain, etc., esta técnica también busca maximizar su precisión, y la forma más sencilla es siempre decir que no hay enlaces. Por ejemplo, si hay 1000 nodos, habrá
$\frac{(1000)(999)}{2} = 499500$ enlaces posibles, pero quizá solo existan $40,000$. Si el modelo decide no crear ningún enlace, tendría una precisión del:
$$\text{Accuracy} = \frac{\text{Predicciones Correctas}}{\text{Total de Casos}} = \frac{459,500}{499,500} = 0.9199$$
Pero aunque sea muy buena métrica, no nos sirve de nada ese modelo.

Utiliza un **Graph Auto-Encoder** que mapea cada $v_i \in V$ a un vector de valores reales. Después se usa un **Decoder** para reconstruir las redes desde su representación vectorial. Este decoder aplica un producto punto entre dos nodos **por cada enlace**

# [PyG (Python Geometric Library](https://pytorch-geometric.readthedocs.io/)

Es una librería construida sobre pytorch para entrenar *Graph Neural Networks*

Aplicaciones de GNN:

- Comercio y venta minorista: interacciones entre usuarios y productos / anuncios, dinámicas de compra y pedidos.

- Salud: interacciones biológicas entre medicamentos, proteínas, vías metabólicas, efectos secundarios.

- Finanzas y Seguros: transacciones financieras entre entidades.

- Transporte: redes de tráfico y logística.

- Manufactura: interacciones en la cadena de suministro y de valor, sistemas de dinámica de fluidos/mecánicos.

- Redes Sociales: redes profesionales, plataformas de redes sociales, plataformas de comunicación.

conjunto de datos **MovieLens** recopilado por el grupo de investigación **GroupLens**. Este conjunto de datos de prueba (toy dataset) describe las calificaciones y la actividad de etiquetado de MovieLens. El conjunto de datos contiene aproximadamente 100 mil calificaciones de más de 9 mil películas hechas por más de 600 usuarios. Vamos a usar este conjunto de datos para generar dos tipos de nodos que contendrán datos para películas y usuarios, respectivamente, y un tipo de arista (enlace) que conectará a usuarios y películas, representando la relación de si un usuario ha calificado una película específica.

La tarea de predicción de enlaces intenta predecir las calificaciones faltantes y puede, por ejemplo, usarse para recomendar películas nuevas a los usuarios.

In [4]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
extract_zip(download_url(url, '.'), '.')

movies_path = './ml-latest-small/movies.csv'
ratings_path = './ml-latest-small/ratings.csv'

Using existing file ml-latest-small.zip
Extracting ./ml-latest-small.zip


In [5]:
movies_df = pd.read_csv(movies_path, index_col='movieId')
movies_df.head()

,title,genres
movieId,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy


`ratings_csv` conecta a los usuarios (dados por userId) y las películas (dadas por movieId)

In [6]:
ratings_df  = pd.read_csv(ratings_path)
ratings_df .head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
#Separamos los géneros para definir de cada pelicula, a cual genero pertenecen
genres = movies_df['genres'].str.get_dummies('|')
print(genres[["Action", "Adventure", "Drama", "Horror"]].head())

movie_feat = torch.from_numpy(genres.values).to(torch.float)
assert movie_feat.size() == (9742, 20)  #20 generos en total.

         Action  Adventure  Drama  Horror
movieId                                  
1             0          1      0       0
2             0          1      0       0
3             0          0      0       0
4             0          0      1       0
5             0          0      0       0


In [8]:
genres.columns

Index(['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller',
       'War', 'Western'],
      dtype='object')

In [9]:
#Crear un mapeo unico por usuario [0, num_user_nodes):
unique_user_id = ratings_df['userId'].unique()
unique_user_id = pd.DataFrame(data={
    'userId': unique_user_id,
    'mappedID': pd.RangeIndex(len(unique_user_id)),
})
print("Mapeamos IDs a valores consecutivos:")
print("==========================================")
print(unique_user_id.head(), "\n")

#Crear un mapeo unico por pelicula [0, num_movie_nodes):
unique_movie_id = ratings_df['movieId'].unique()
unique_movie_id = pd.DataFrame(data={
    'movieId': unique_movie_id,
    'mappedID': pd.RangeIndex(len(unique_movie_id)),
})
print("Mapeo de ID de peliculas a valores consecutivos:")
print("===========================================")
print(unique_movie_id.head())
#Unimos para obtener los enlaces entre usuarios y peliculas
ratings_user_id = pd.merge(ratings_df['userId'], unique_user_id,
                            left_on='userId', right_on='userId', how='left')
ratings_user_id = torch.from_numpy(ratings_user_id['mappedID'].values)
ratings_movie_id = pd.merge(ratings_df['movieId'], unique_movie_id,
                            left_on='movieId', right_on='movieId', how='left')
ratings_movie_id = torch.from_numpy(ratings_movie_id['mappedID'].values)
#Con esto, estamos listos para construir nuestro edge_index en formato COO,
#siguiendo la semántica de PyG:
edge_index_user_to_movie = torch.stack([ratings_user_id, ratings_movie_id], dim=0)
assert edge_index_user_to_movie.size() == (2, 100836)
print()
print("Final edge indices pointing from users to movies:")
print("=================================================")
print(edge_index_user_to_movie)

Mapeamos IDs a valores consecutivos:
   userId  mappedID
0       1         0
1       2         1
2       3         2
3       4         3
4       5         4 

Mapeo de ID de peliculas a valores consecutivos:
   movieId  mappedID
0        1         0
1        3         1
2        6         2
3       47         3
4       50         4

Final edge indices pointing from users to movies:
tensor([[   0,    0,    0,  ...,  609,  609,  609],
        [   0,    1,    2,  ..., 3121, 1392, 2873]])


El formato COO (Coordinate format) es una forma de representar matrices dispersas (sparse matrices). En el contexto de grafos y PyG, significa que el edge_index se guarda como dos filas de números:

La primera fila contiene los índices de los nodos de origen (por ejemplo, los usuarios).

La segunda fila contiene los índices de los nodos de destino (las películas).

`[5, 12]`, significa que hay un enlace (o calificación) que va desde el usuario 5 hacia la película 12.

Nos encargamos de añadir las aristas inversas (`reverse edges`) al objeto `HeteroData`. Esto permite que nuestro modelo GNN utilice ambas direcciones de la arista para el paso de mensajes (message passing).

In [10]:
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

In [11]:
data = HeteroData()
#Guardar los índices de los nodos:
data["user"].node_id = torch.arange(len(unique_user_id))
data["movie"].node_id = torch.arange(len(movies_df))

#Añadir las características de los nodos e índices de las aristas:
data["movie"].x = movie_feat
data["user", "rates", "movie"].edge_index = edge_index_user_to_movie

#Aseguramos de añadir las aristas inversas de películas a usuarios
#para permitir que una GNN pueda pasar mensajes en ambas direcciones.
#Podemos aprovechar la transformación `T.ToUndirected()` de PyG para esto:
data = T.ToUndirected()(data)

In [12]:
data

HeteroData(
  user={ node_id=[610] },
  movie={
    node_id=[9742],
    x=[9742, 20],
  },
  (user, rates, movie)={ edge_index=[2, 100836] },
  (movie, rev_rates, user)={ edge_index=[2, 100836] }
)

Dado que nuestros datos ya están listos para usarse, podemos dividir las calificaciones de los usuarios en conjuntos de **entrenamiento**, **validación** y **test** para garantizar que no filtremos información sobre las aristas utilizadas durante la evaluación hacia la fase de entrenamiento. Para esto, utilizamos la transformación `transforms.RandomLinkSplit` de PyG.

Esta transforma y divide aleatoriamente las aristas en ("user", "rates", "movie") en aristas de entrenamiento, validación y prueba. El parámetro `disjoint_train_ratio` separa aún más las aristas del conjunto de entrenamiento en:
- Aristas utilizadas para el paso de mensajes (edge_index).
- Aristas utilizadas para la supervisión (edge_label_index).

También debemos especificar el tipo de arista inversa ("movie", "rev_rates", "user"). Esto permite que `RandomLinkSplit` elimine las aristas inversas correspondientes para **no filtrar información** en la fase de entrenamiento.

In [13]:
#Para esto, primero dividimos el conjunto de aristas en
#entrenamiento (80%), validación (10%) y prueba (10%).

#En las aristas de entrenamiento, usamos el 70% para paso de mensajes
#y el 30% para supervisión.

#Además, queremos generar aristas negativas fijas para evaluación.
#Las aristas negativas durante el entrenamiento se generarán sobre la marcha.
#Usamos la transformación `RandomLinkSplit()` de PyG para esto:
transform = T.RandomLinkSplit(
    num_val=0.1,               #de datos de validacion
    num_test=0.1,              #% de datos para testing
    disjoint_train_ratio=0.3,  #30% supervision
    neg_sampling_ratio=2.0,    #por cada enlace genera 2 falsos para entrenar
    add_negative_train_samples=False, #Enlaces q no existen (que peliculas no clasificó)!!!!!!!!!!!!,
                                      #no genera de golpe todas para evitar colapsar la memoria
    edge_types=("user", "rates", "movie"), #que va a entrenar? Usuario calificando pelis
    rev_edge_types=("movie", "rev_rates", "user"), #oculta las aristas inversas para no hacer data leakage
)

#Ejecutamos la transformación para obtener los tres conjuntos de datos
train_data, val_data, test_data = transform(data)

Creamos un cargador por mini-lotes que generará subgrafos, los cuales podrán usarse como entrada para nuestra GNN. Aunque este paso **no es estrictamente necesario** para grafos a pequeña escala, es absolutamente indispensable para aplicar GNNs en grafos más grandes que, de otro modo, no cabrían en la memoria de la GPU.

Aquí, hacemos uso de `loader.LinkNeighborLoader`, que realiza un muestreo de múltiples saltos desde ambos extremos de un enlace y crea un subgrafo a partir de ellos. `edge_label_index` sirve como los "enlaces semilla" desde los cuales se comienza el muestreo.

In [12]:
!pip install pyg-lib -f https://data.pyg.org/whl/torch-2.3.0+cu121.html --quiet

In [14]:
from torch_geometric.loader import LinkNeighborLoader

In [15]:
#Durante el entrenamiento, queremos muestrear aristas negativas sobre la marcha
#con una proporción de 2:1.
#Podemos hacer uso del `loader.LinkNeighborLoader` de PyG:

#Definir las aristas semilla:
edge_label_index = train_data["user", "rates", "movie"].edge_label_index
edge_label = train_data["user", "rates", "movie"].edge_label

train_loader = LinkNeighborLoader(
    data=train_data,
    #En el primer salto, muestreamos como máximo 20 vecinos.
    #En el segundo salto, muestreamos como máximo 10 vecinos.
    num_neighbors=[20, 10], #Muestreo de vecinos por capas (saltos)
    neg_sampling_ratio=2.0, #2 ejemplos negativos por cada positivo
    edge_label_index=(("user", "rates", "movie"), edge_label_index),
    edge_label=edge_label,
    batch_size=128, #Tamaño del lote
    shuffle=True, #Mezclar los datos en cada época
)

El modelo GNN aprenderá representaciones enriquecidas de los nodos a partir de los subgrafos circundantes, las cuales pueden usarse después para derivar **predicciones a nivel de arista**. Para definir nuestra GNN heterogénea, utilizamos `nn.SAGEConv` y la función `nn.to_hetero()`, la cual transforma una GNN definida para grafos homogéneos para que pueda aplicarse en grafos heterogéneos.

Definimos un clasificador a nivel de enlace, que toma los *embeddings* (representaciones numéricas) de ambos nodos del enlace que intentamos predecir y aplica un **producto punto** entre ellos.

Dado que los usuarios no tienen información previa a nivel de nodo (características), elegimos aprender sus rasgos mediante una capa `torch.nn.Embedding`.

In [16]:
from torch_geometric.nn import SAGEConv, to_hetero
import torch.nn.functional as F
from torch import Tensor

In [17]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        #Aplicamos la primera convolución y una activación ReLU
        x = F.relu(self.conv1(x, edge_index))
        #Segunda convolución para refinar la representación
        x = self.conv2(x, edge_index)
        return x

#Nuestro clasificador final aplica el producto punto entre los embeddings
#de los nodos origen y destino para derivar predicciones a nivel de arista:
class Classifier(torch.nn.Module):
    def forward(self, x_user: Tensor, x_movie: Tensor, edge_label_index: Tensor) -> Tensor:
        #Convertimos los embeddings de los nodos en representaciones a nivel de arista:
        edge_feat_user = x_user[edge_label_index[0]]
        edge_feat_movie = x_movie[edge_label_index[1]]
        #Aplicamos producto punto para obtener una predicción por cada arista:
        return (edge_feat_user * edge_feat_movie).sum(dim=-1)

class Model(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        #Como el dataset no tiene muchas características, aprendemos matrices de embeddings
        #para usuarios y películas:
        self.movie_lin = torch.nn.Linear(20, hidden_channels)
        self.user_emb = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.movie_emb = torch.nn.Embedding(data["movie"].num_nodes, hidden_channels)

        #Instanciamos la GNN homogénea:
        self.gnn = GNN(hidden_channels)

        #Convertimos el modelo GNN en una variante heterogénea:
        #Esto hace que el modelo aprenda pesos distintos para cada tipo de relación
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())
        self.classifier = Classifier()

    def forward(self, data: HeteroData) -> Tensor:
        #Creamos el diccionario de características iniciales:
        #Para películas combinamos sus géneros (movie_lin) con su ID aprendido (movie_emb)
        x_dict = {
          "user": self.user_emb(data["user"].node_id),
          "movie": self.movie_lin(data["movie"].x) + self.movie_emb(data["movie"].node_id),
        }

        #`x_dict` contiene las matrices de características de todos los tipos de nodo
        #`edge_index_dict` contiene todos los índices de aristas de todos los tipos
        x_dict = self.gnn(x_dict, data.edge_index_dict)

        #Predecimos la probabilidad de existencia de enlaces
        pred = self.classifier(
            x_dict["user"],
            x_dict["movie"],
            data["user", "rates", "movie"].edge_label_index,
        )
        return pred

model = Model(hidden_channels=64)

In [18]:
model

Model(
  (movie_lin): Linear(in_features=20, out_features=64, bias=True)
  (user_emb): Embedding(610, 64)
  (movie_emb): Embedding(9742, 64)
  (gnn): GraphModule(
    (conv1): ModuleDict(
      (user__rates__movie): SAGEConv(64, 64, aggr=mean)
      (movie__rev_rates__user): SAGEConv(64, 64, aggr=mean)
    )
    (conv2): ModuleDict(
      (user__rates__movie): SAGEConv(64, 64, aggr=mean)
      (movie__rev_rates__user): SAGEConv(64, 64, aggr=mean)
    )
  )
  (classifier): Classifier()
)

# Entrenamiento de la GNN

In [20]:
import tqdm
import torch.nn.functional as F
import os

In [21]:
#Configuramos el dispositivo: usar GPU (cuda) si está disponible, si no, CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: '{device}'")
#Movemos el modelo al dispositivo seleccionado
model = model.to(device)

#Configuramos el optimizador Adam con una tasa de aprendizaje (lr) de 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#Entrenamos durante 20 épocas (iteraciones completas sobre el dataset)
for epoch in range(1, 21):
    total_loss = total_examples = 0

    #Iteramos sobre los mini-lotes generados por el train_loader
    for sampled_data in tqdm.tqdm(train_loader):
        optimizer.zero_grad() #Reiniciamos los gradientes

        sampled_data.to(device) #Movemos el subgrafo actual a la GPU/CPU
        pred = model(sampled_data) #Paso hacia adelante: predicción

        #Obtenemos las etiquetas reales (1 si existe el enlace, 0 si no)
        ground_truth = sampled_data["user", "rates", "movie"].edge_label

        #Calculamos la pérdida comparando predicción vs realidad
        #BCEWithLogitsLoss es ideal para problemas de clasificación binaria (si/no)
        loss = F.binary_cross_entropy_with_logits(pred, ground_truth)

        loss.backward() #Retropropagación: calcular cuánto se equivocó cada neurona
        optimizer.step() #Actualizar los pesos del modelo

        total_loss += float(loss) * pred.numel()
        total_examples += pred.numel()

    print(f"Epoch: {epoch:03d}, Loss: {total_loss / total_examples:.4f}")

Device: 'cpu'


100%|██████████| 190/190 [00:05<00:00, 32.94it/s]


Epoch: 001, Loss: 0.2897


100%|██████████| 190/190 [00:05<00:00, 32.48it/s]


Epoch: 002, Loss: 0.2816


...

Epoch: 018, Loss: 0.1934


100%|██████████| 190/190 [00:06<00:00, 31.58it/s]


Epoch: 019, Loss: 0.1920


100%|██████████| 190/190 [00:04<00:00, 41.56it/s]

Epoch: 020, Loss: 0.1895


La *Loss Function* (función de pérdida) representa qué tan equivocado está el modelo, entre menor sea, quiere decir que más entiende la estructura de la red

Después del entrenamiento, evaluamos nuestro modelo en datos no vistos del conjunto de validación. Para esto, definimos un nuevo `LinkNeighborLoader` (que ahora itera sobre las aristas en el conjunto de validación), obtenemos las predicciones sobre las aristas de validación ejecutando el modelo, y finalmente evaluamos el rendimiento del modelo calculando la puntuación **AUC (Área Bajo la Curva)** de la funcinón ROC (*Receiver Operating Characteristic*) sobre el conjunto de predicciones y sus correspondientes aristas reales o ground-truth (incluyendo tanto aristas positivas como negativas).

In [26]:
#Definir las aristas semilla de validación:
edge_label_index = val_data["user", "rates", "movie"].edge_label_index
edge_label = val_data["user", "rates", "movie"].edge_label

#Creamos el loader para los datos de validación
val_loader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=[20, 10],
    edge_label_index=(("user", "rates", "movie"), edge_label_index),
    edge_label=edge_label,
    batch_size=3 * 128,
    shuffle=False, #No necesitamos mezclar los datos en la evaluación
)

sampled_data = next(iter(val_loader))

Es un solo mini-lote (un subgrafo) generado por `LinkNeighborLoader`. Meter todo el grafo a la vez era imposible, así que se hace por batches.

El *Loader* capturó 605 usuarios y 2635 películas.

----

`x=[2635, 20]`: Son las características de las películas. 2678 películas en este lote, y cada una tiene un vector de tamaño 20 (que son los géneros de peliculas).

`node_id` y `n_id`: Listas que guardan los "IDs originales". Como este es un subgrafo, el usuario que aquí es el número "0", en el dataset completo podría ser el usuario "452".

----
`(user, rates, movie) = {...}`: De todo este lote, el modelo va a intentar predecir 297 enlaces.

`edge_label_index`: Contiene los IDs de los usuarios y las películas de esos 297 enlaces que queremos clasificar.

`edge_label`: Son las respuestas correctas. Contiene 297 números que son 1 (el enlace es real) o 0 (el enlace es falso, creado aleatoriamente por `neg_sampling_ratio=2.0`). El modelo las usa para calcular el error (Loss).

-----

`(movie, rev_rates, user)={...}`

Para clasificar los 297 enlaces, el modelo necesita contexto. `edge_index` contiene 19,309 conexiones reales que el modelo usa para "pasar mensajes" entre usuarios y películas para entender sus gustos.

El modelo usa estos 19,309 enlaces para entrenar y luego se le hace un test sobre los 297 enlaces de edge_label.

In [31]:
sampled_data

HeteroData(
  user={
    node_id=[605],
    n_id=[605],
    num_sampled_nodes=[3],
  },
  movie={
    node_id=[2635],
    x=[2635, 20],
    n_id=[2635],
    num_sampled_nodes=[3],
  },
  (user, rates, movie)={
    edge_index=[2, 16809],
    edge_label=[297],
    edge_label_index=[2, 297],
    e_id=[16809],
    num_sampled_edges=[2],
    input_id=[297],
  },
  (movie, rev_rates, user)={
    edge_index=[2, 6696],
    e_id=[6696],
    num_sampled_edges=[2],
  }
)

In [28]:
from sklearn.metrics import roc_auc_score

preds = []
ground_truths = []

#Iteramos sobre el cargador de validación
for sampled_data in tqdm.tqdm(val_loader):
    #`torch.no_grad()` apaga el motor de aprendizaje para ahorrar memoria y hacer el proceso más rápido
    with torch.no_grad():
        sampled_data.to(device)
        preds.append(model(sampled_data))
        ground_truths.append(sampled_data["user", "rates", "movie"].edge_label)

#Juntamos todas las predicciones y etiquetas en un solo tensor y las pasamos a la CPU
pred = torch.cat(preds, dim=0).cpu().numpy()
ground_truth = torch.cat(ground_truths, dim=0).cpu().numpy()

#Calculamos la métrica final
auc = roc_auc_score(ground_truth, pred)

print()
print(f"Validation AUC: {auc:.4f}")

100%|██████████| 79/79 [00:00<00:00, 103.83it/s]


Validation AUC: 0.9287


En el problema de predicción de enlaces (Link Prediction) con el dataset de MovieLens, el modelo tenía que predecir si un usuario calificaría una película o no.

Si se toma un usuario al azar y se le pide al modelo que analice dos películas simultáneamente:

- Una película que calificó en la vida real (un enlace positivo).
- Una película que NUNCA ha visto ni calificado (un enlace negativo).

El AUC de 0.9287 significa que hay un 92.87% de probabilidad de que el modelo le asigne una puntuación matemática más alta a la película correcta que a la incorrecta.

In [29]:
pred

array([-3.0563078, -1.6330155, -3.522672 , ..., -1.8237717, -6.945486 ,
       -8.044142 ], dtype=float32)

In [30]:
ground_truth

array([1., 1., 1., ..., 0., 0., 0.], dtype=float32)

[Medium](https://medium.com/@pytorch_geometric/link-prediction-on-heterogeneous-graphs-with-pyg-6d5c29677c70)